## Easy: Fixed XOR


In [34]:
import sys

def xor(a, b):
    a = bytes.fromhex(a)
    b = bytes.fromhex(b)
    if len(a) != len(b):
        print("buffers must be equal length")
        sys.exit(1)
    res = bytearray()
    for x, y in zip(a, b):
        res.append(x ^ y)
    return res.hex()

a = "1c0111001f010100061a024b53535009181c"
b = "686974207468652062756c6c277320657965"
assert(xor(a, b) == "746865206b696420646f6e277420706c6179")
xor(a, b)

'746865206b696420646f6e277420706c6179'

## Medium: Detect AES in ECB mode
The same 16 byte plaintext block will always produce the same 16 byte ciphertext.

All ciphertexts are the same size, so the one with the fewest different blocks is most likely the one encrypted with ECB.

In [35]:
!curl https://cryptopals.com/static/challenge-data/8.txt > 8.txt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 65484  100 65484    0     0   324k      0 --:--:-- --:--:-- --:--:--  326k


In [36]:
with open("8.txt", "r") as f:
    lines = [line.strip() for line in f.readlines()]

num_different_blocks = []

for i, line in enumerate(lines):
    data = bytes.fromhex(line)
    blocks = [data[i:i + 16] for i in range(0, len(data), 16)]
    num_different_blocks.append((len(set(blocks)), i))

num_different_blocks.sort()
print("Most likely ECB lines:")
for i in range(3):
    print(f"line {num_different_blocks[i][1]} has {num_different_blocks[i][0]} different blocks")

Most likely ECB lines:
line 132 has 7 different blocks
line 0 has 10 different blocks
line 1 has 10 different blocks


So we can confidently say that ciphertext number 132 (counting from zero, or 133 if counting from one) is the ECB-encrypted

## Hard: Clone an MT19937 RNG from its output
624 ints are generated by original RNG and saved into a file

In [37]:
import random

rng = random.Random(12345)
generated = [rng.getrandbits(32) for _ in range(624)]
with open("generated.txt", "w") as f:
    for x in generated:
        f.write(f"{x}\n")

The only thing that attacker gets is this file. They untemper the numbers and recreate the state.

In [38]:
def unright(y, shift):
    x = y
    for _ in range(32 // shift + 1):
        x = y ^ (x >> shift)
    return x

def unleft(y, shift, mask):
    x = y
    for _ in range(32 // shift + 1):
        x = y ^ ((x << shift) & mask)
    return x

def untemper(y):
    y = unright(y, 18)
    y = unleft(y, 15, 0xEFC60000)
    y = unleft(y, 7, 0x9D2C5680)
    y = unright(y, 11)

    return y

with open("generated.txt", "r") as f:
    generated = [int(line.strip()) for line in f.readlines()]
untempered = [untemper(x) for x in generated]
state = (3, tuple(untempered + [624]), None)
cloned_rng = random.Random()
cloned_rng.setstate(state)


And we can check that now the attacker clones what original RNG generates

In [39]:
for i in range(10):
    o = rng.getrandbits(32)
    c = cloned_rng.getrandbits(32)
    print(f"{i:2d} | original: {o:10d} | clone: {c:10d} | same? {o == c}")

 0 | original: 4171722749 | clone: 4171722749 | same? True
 1 | original: 3649179348 | clone: 3649179348 | same? True
 2 | original: 3014839245 | clone: 3014839245 | same? True
 3 | original: 2072524753 | clone: 2072524753 | same? True
 4 | original: 3937770851 | clone: 3937770851 | same? True
 5 | original: 2787122923 | clone: 2787122923 | same? True
 6 | original:  681451109 | clone:  681451109 | same? True
 7 | original: 3862615832 | clone: 3862615832 | same? True
 8 | original: 1812468063 | clone: 1812468063 | same? True
 9 | original:  204033643 | clone:  204033643 | same? True
